# Queues and Deques

```{contents}
:local:
:depth: 2
```


```{index} Separate Logical Order from the Positions That Store It
```



```{index} Separate Logical Order from the Positions That Store It
```

## Separate Logical Order from the Positions That Store It



In a ring queue, removing the front does not require shifting every later value. A front index and Count identify the logical sequence within fixed circular storage. With capacity four, front at three and Count two, the live positions are three and zero. The next insertion belongs at (front + Count) modulo capacity, which is position one. Raw left-to-right array printing would not show service order.

| Logical position | Physical position when front is 3 and capacity is 4 |
| --- | --- |
| First | 3 |
| Second | 0 |
| Next insertion | 1 |

Count distinguishes full from empty when the wrapped insertion position coincides with front. Element values cannot safely act as empty markers because zero or null may be legitimate stored values under the contract. A bounded queue rejects overflow before writing; a growing queue allocates new storage and copies logical order to consecutive positions before resetting front.

A deque permits both ends, so it needs additional endpoint/update reasoning. It does not supply arbitrary middle indexing or priority selection merely because it is more flexible than a queue. Guided trace: enqueue three entries, remove two, then enqueue enough to wrap. Write logical order after every step and compare it with physical slots. Repeat the exercise after a growth copy; the representation changes while service order must remain the same.

For related explanation, see the further reading below.[^development-reading]


```{index} queue; FIFO
```



```{index} Queue Behavior
```

## Queue Behavior



A **queue** is an abstract data type where the earliest added item is the next item removed. This access rule is called **first in, first out**, or **FIFO**.

The core operations are:

- `Enqueue(value)` adds a value to the back.
- `Dequeue()` removes and returns the front value.
- `Peek()` returns the front value without removing it.
- `Count` reports how many values are waiting.

The vocabulary mirrors the stack's: items are *enqueued* at the back, also called the rear, and *dequeued* from the front. Only the front is readable, so a queue, like a stack, is a list with deliberately restricted access. The restriction is what makes it useful. When a program needs first-come, first-served behavior, a queue type states that policy in the code and makes the alternative, reaching into the middle, impossible rather than merely discouraged. The example uses the library's `Queue<T>`, which grows as needed; the sections below build bounded and linked versions to show what the library hides.

In [ ]:
using System;
using System.Collections.Generic;

var serviceLine = new Queue<string>();
serviceLine.Enqueue("Amina");
serviceLine.Enqueue("Diego");
serviceLine.Enqueue("Mina");

Console.WriteLine($"Serving: {serviceLine.Dequeue()}");
Console.WriteLine($"Next: {serviceLine.Peek()}");
Console.WriteLine($"Still waiting: {serviceLine.Count}");


Serving: Amina
Next: Diego
Still waiting: 2


The first customer enqueued is the first customer served. Unlike a list, a queue communicates that callers should not pick arbitrary positions from the middle.


```{index} queue; worklist
```



```{index} Worklists and Scheduling
```

## Worklists and Scheduling



Queues often represent work that should be processed in arrival order: help-desk tickets, print jobs, messages, or search states. This pattern is called a **worklist** because the collection stores work that remains to be done.

A worklist loop has a fixed shape: take the next item from the front, do its work, and enqueue any new work that it reveals. Because new items always join the back, work discovered later never jumps ahead of work that was already waiting. That property is what later chapters rely on for breadth-first traversal, where the queue guarantees that every item at distance one is processed before any item at distance two. The example processes three fixed steps; the breadth-first example in 15.3 adds the discovery of new work inside the loop.

In [ ]:
using System;
using System.Collections.Generic;

var work = new Queue<string>();
work.Enqueue("Validate order");
work.Enqueue("Charge card");
work.Enqueue("Send receipt");

int step = 1;
while (work.Count > 0)
{
    string task = work.Dequeue();
    Console.WriteLine($"Step {step}: {task}");
    step++;
}


Step 1: Validate order
Step 2: Charge card
Step 3: Send receipt


A queue makes the scheduling policy explicit: new work joins the back, and existing work leaves from the front.


```{index} ring buffer; queue implementation
```



```{index} Ring Buffers
```

## Ring Buffers



An array-backed queue cannot simply remove index `0` by shifting every element left. That would make each dequeue expensive. A **ring buffer** solves this by letting the front and back indexes wrap around the array.

For a bounded queue, the important state is:

- `front`: the index of the next item to remove
- `back`: the index where the next item will be added
- `count`: the number of stored items

Why the wrap is necessary becomes clear from the naive version. Suppose `front` simply moves right on every dequeue and `back` moves right on every enqueue. After a few hundred operations both indexes sit near the end of the array while the slots behind `front` are empty and unusable: the queue "drifts" toward the end and reports itself full with most of its storage free. Shifting everything left after each dequeue repairs the drift but costs O(n) per operation. Letting the indexes wrap to slot 0 with the modulo operator repairs it for free, which is why the physical layout becomes a ring.

The example keeps the three state variables as top-level locals so that each operation's effect on them is visible. It fills three slots, serves one, and then enqueues two more, which forces `back` to wrap past the end of the four-slot array.

In [ ]:
using System;

int capacity = 4;
int front = 0;
int back = 0;
int count = 0;

void Enqueue(string value, string[] items)
{
    if (count == items.Length)
    {
        throw new InvalidOperationException("Queue is full.");
    }

    items[back] = value;
    back = (back + 1) % items.Length;
    count++;
}

string Dequeue(string[] items)
{
    if (count == 0)
    {
        throw new InvalidOperationException("Queue is empty.");
    }

    string value = items[front];
    items[front] = null!;
    front = (front + 1) % items.Length;
    count--;
    return value;
}

var items = new string[capacity];
Enqueue("A", items);
Enqueue("B", items);
Enqueue("C", items);
Console.WriteLine(Dequeue(items));
Enqueue("D", items);
Enqueue("E", items);

Console.WriteLine($"front={front}, back={back}, count={count}");


A
front=1, back=1, count=4


The modulo operation wraps an index back to zero after it reaches the end of the array. This avoids shifting values after every dequeue.


After the five operations, `front` is 1 and `back` is 1, and the two indexes are equal even though the queue is full. The physical array holds `E, B, C, D`: `B` is the oldest item, at slot 1, and `E`, the newest, wrapped around to slot 0. A loop that printed the array from left to right would misreport the service order, which is the point made at the top of this page. The `count` variable is what lets `Enqueue` reject a fifth item; without it, the state `front == back` is ambiguous, as the next subsection explains.

```{index} Why Front and Back Alone Cannot Tell Full from Empty
```

### Why Front and Back Alone Cannot Tell Full from Empty

A ring with n slots can hold 0, 1, 2, ..., n items: that is n + 1 distinct states the implementation must tell apart. If `front` is held fixed, the only thing that varies is `back`, and `back` can take only n values, one per slot. Mapping n + 1 states onto n values forces two states to share a value; this is the pigeonhole principle, and the two states that collide are empty and full, both of which leave `back` equal to `front`. No cleverness with the two indexes alone escapes this.

There are two standard repairs, and both appear in real libraries.

| Repair | How it works | Cost |
| --- | --- | --- |
| Keep a count | `count` is the third state variable; full is `count == Capacity`, empty is `count == 0` | One integer; `back` can even be derived as `(front + count) % Capacity` |
| Leave one slot unused | Allocate n + 1 slots for a capacity of n; full is "the next insertion slot equals `front`", empty is `back == front` | One wasted slot; no count field |

The `RingQueue<T>` later on this page uses the first repair and derives `back` from `front` and `Count`, so there is no third index to keep consistent. Either repair is correct; what is not correct is testing the stored values, because `null` or zero may be legitimate data.

```{index} deque
```



```{index} Deques
```

## Deques



A **deque**, pronounced like "deck", is a double-ended queue. It allows adding and removing at both the front and the back. That makes it more flexible than a stack or queue, but still more controlled than arbitrary list indexing.

C# does not provide a dedicated `Deque<T>` in the base class library. For small examples, `LinkedList<T>` can demonstrate double-ended operations.

The four operations usually have paired names: add first and add last, remove first and remove last. A deque that is only ever used at one end behaves as a stack; one that adds at the back and removes at the front behaves as a queue. That is why some libraries provide a deque as the single general type and let the program choose which ends to use. The restriction a deque keeps is that nothing happens in the middle: no indexing, no insertion between two elements. The example uses `LinkedList<T>`, whose `AddFirst`, `AddLast`, `RemoveFirst`, and `RemoveLast` are all constant time because it is doubly linked and keeps both endpoints.

In [ ]:
using System;
using System.Collections.Generic;

var deque = new LinkedList<string>();
deque.AddLast("middle");
deque.AddFirst("front");
deque.AddLast("back");

Console.WriteLine(deque.First!.Value);
Console.WriteLine(deque.Last!.Value);

deque.RemoveFirst();
deque.RemoveLast();

Console.WriteLine($"Remaining: {deque.First!.Value}");


front
back
Remaining: middle


A deque is useful when both ends have meaning, such as browser history, task stealing, or algorithms that need to expand from the front while sometimes adding urgent items to the front.


```{index} trace wraparound and distinguish full from empty
```



```{index} Trace Wraparound and Distinguish Full from Empty
```

## Trace Wraparound and Distinguish Full from Empty



A ring buffer stores logical order across the end of physical storage. Start with capacity four, `front = back = count = 0`. Back is the next insertion slot, not the last occupied slot.

| Operation | Slots 0, 1, 2, 3 | Front | Back | Count | Logical order |
| --- | --- | ---: | ---: | ---: | --- |
| Enqueue A, B, C | A, B, C, unused | 0 | 3 | 3 | A, B, C |
| Dequeue | unused, B, C, unused | 1 | 3 | 2 | B, C |
| Enqueue D | unused, B, C, D | 1 | 0 | 3 | B, C, D |
| Enqueue E | E, B, C, D | 1 | 1 | 4 | B, C, D, E |
| Dequeue B | E, unused, C, D | 2 | 1 | 3 | C, D, E |

`front == back` occurs when the buffer is empty and when it is full. Count distinguishes those states. For logical position i, the physical slot is `(front + i) % Capacity`. The invariants are positive capacity, a front index inside the array, and `0 <= Count <= Capacity`. The next insertion slot is `(front + Count) % Capacity`.

**Trace:** Continue until empty, then enqueue F. Give every front, back, and count. Explain why physical slot zero does not always hold the oldest item.


```{index} a complete bounded generic queue
```



```{index} A Complete Bounded Generic Queue
```

## A Complete Bounded Generic Queue



This implementation derives back from front and count, so it does not maintain a redundant third state variable. Enqueue checks for full storage before writing; dequeue checks emptiness before advancing front. Both clear or fill one slot and update count once.

Capacity is fixed. A full enqueue throws and preserves the waiting tickets. That overflow policy is part of the contract; silently overwriting the oldest ticket would implement different behavior. Capacity one is legal and exposes full/empty mistakes quickly.

**Test:** Fill a capacity-one queue, reject a second enqueue, remove the original item, reject an empty dequeue, and reuse it. Verify returned values and count after each request.


In [ ]:
using System;
var tickets = new RingQueue<string>(4);
tickets.Enqueue("A"); tickets.Enqueue("B"); tickets.Enqueue("C");
Console.WriteLine($"served={tickets.Dequeue()}");
tickets.Enqueue("D"); tickets.Enqueue("E");
Console.WriteLine($"count={tickets.Count}, front={tickets.Peek()}");
while (tickets.TryDequeue(out string ticket)) Console.WriteLine(ticket);
tickets.Enqueue("F"); Console.WriteLine($"reused={tickets.Dequeue()}");

public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

served=A
count=4, front=B
B
C
D
E
reused=F


The first line reports `served=A`; the second shows `count=4` with `B` at the front after `D` and `E` wrapped into the freed slot. The `TryDequeue` loop then drains `B`, `C`, `D`, `E` in arrival order even though they sit in physical slots 1, 2, 3, 0, and `reused=F` confirms that an emptied ring accepts new work. `Peek` is the single place that checks emptiness, and `Dequeue` reuses it, so the empty-queue policy lives in one line rather than two.

```{index} Code Completion: Dequeue from a Ring
```

### Code Completion: Dequeue from a Ring

`Enqueue` is written; `Dequeue` is not. Complete it so that it removes and returns the front item, clears the freed slot, advances `front` with wraparound, and throws `InvalidOperationException` on an empty queue. The expected output is `A`, `B`, then `count=2, front=C`, then `E` after the wrap, and finally `empty queue rejected`.

In [ ]:
using System;

var q = new Ring<string>(3);
q.Enqueue("A"); q.Enqueue("B"); q.Enqueue("C");
Console.WriteLine(q.Dequeue());
Console.WriteLine(q.Dequeue());
q.Enqueue("D"); q.Enqueue("E");               // E wraps into slot 1
Console.WriteLine($"count={q.Count}, front={q.Dequeue()}");
q.Dequeue();
Console.WriteLine(q.Dequeue());
try { q.Dequeue(); } catch (InvalidOperationException) { Console.WriteLine("empty queue rejected"); }

public class Ring<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public Ring(int capacity) => items = new T[capacity];

    public void Enqueue(T value)
    {
        if (Count == items.Length) throw new InvalidOperationException("Queue is full.");
        items[(front + Count) % items.Length] = value;
        Count++;
    }

    public T Dequeue()
    {
        // Your code starts here.
        throw new NotImplementedException();
        // Your code ends here.
    }
}

In [ ]:
// Solution
using System;

var q = new Ring<string>(3);
q.Enqueue("A"); q.Enqueue("B"); q.Enqueue("C");
Console.WriteLine(q.Dequeue());
Console.WriteLine(q.Dequeue());
q.Enqueue("D"); q.Enqueue("E");               // E wraps into slot 1
Console.WriteLine($"count={q.Count}, front={q.Dequeue()}");
q.Dequeue();
Console.WriteLine(q.Dequeue());
try { q.Dequeue(); } catch (InvalidOperationException) { Console.WriteLine("empty queue rejected"); }

public class Ring<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public Ring(int capacity) => items = new T[capacity];

    public void Enqueue(T value)
    {
        if (Count == items.Length) throw new InvalidOperationException("Queue is full.");
        items[(front + Count) % items.Length] = value;
        Count++;
    }

    public T Dequeue()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        T value = items[front];
        items[front] = default!;                  // do not keep a reference to served work
        front = (front + 1) % items.Length;       // wrap, never shift
        Count--;
        return value;
    }
}

A
B
count=3, front=C
E
empty queue rejected


The four statements after the check must happen in this order: read the value before the slot is cleared, clear before `front` moves on, and move `front` before `Count` is reduced only by convention, since those two are independent. The two mistakes that appear most often are `front++` without the modulo, which runs off the end of the array after three dequeues, and forgetting `Count--`, after which the ring reports itself full forever. Capacity 3 was chosen so that the wrap happens within a few operations; a trace with capacity 1 is the quickest way to find a full-or-empty mistake.

```{index} Code Tracing: Ring State After Each Operation
```

### Code Tracing: Ring State After Each Operation

A `RingQueue<string>` of capacity 3 processes the operations below. Before running the solution, fill in `front`, `Count`, the derived insertion slot `(front + Count) % 3`, and the physical slots 0, 1, 2 after each line.

```csharp
var r = new RingQueue<string>(3);
r.Enqueue("P"); r.Enqueue("Q");
r.Dequeue();
r.Enqueue("R"); r.Enqueue("S");
r.Dequeue(); r.Dequeue();
r.Enqueue("T");
```

Finally, predict the output of `while (r.TryDequeue(out var x)) Console.Write(x + " ");`.

In [ ]:
// Solution
using System;

var r = new RingQueue<string>(3);
Step("enqueue P, Q", () => { r.Enqueue("P"); r.Enqueue("Q"); });
Step("dequeue", () => r.Dequeue());
Step("enqueue R, S", () => { r.Enqueue("R"); r.Enqueue("S"); });
Step("dequeue, dequeue", () => { r.Dequeue(); r.Dequeue(); });
Step("enqueue T", () => r.Enqueue("T"));
Console.Write("drain: ");
while (r.TryDequeue(out var x)) Console.Write(x + " ");
Console.WriteLine();

void Step(string label, Action act)
{
    act();
    Console.WriteLine($"{label,-18} front={r.Front} count={r.Count} next slot={(r.Front + r.Count) % 3} slots=[{r.Slot(0)},{r.Slot(1)},{r.Slot(2)}]");
}

public class RingQueue<T>
{
    private readonly T[] items;
    public int Front { get; private set; }
    public int Count { get; private set; }
    public RingQueue(int capacity) => items = new T[capacity];
    public string Slot(int i) => items[i] is null ? "-" : items[i]!.ToString()!;
    public void Enqueue(T value)
    {
        if (Count == items.Length) throw new InvalidOperationException("Queue is full.");
        items[(Front + Count) % items.Length] = value;
        Count++;
    }
    public T Dequeue()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        T value = items[Front];
        items[Front] = default!;
        Front = (Front + 1) % items.Length;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

enqueue P, Q       front=0 count=2 next slot=2 slots=[P,Q,-]
dequeue            front=1 count=1 next slot=2 slots=[-,Q,-]
enqueue R, S       front=1 count=3 next slot=1 slots=[S,Q,R]
dequeue, dequeue   front=0 count=1 next slot=1 slots=[S,-,-]
enqueue T          front=0 count=2 next slot=2 slots=[S,T,-]
drain: S T 


After `enqueue R, S` the ring is full with `front` at 1 and the slots reading `S, Q, R`: `S` wrapped into slot 0. The two dequeues serve `Q` and `R`, leaving only `S` in slot 0 with `front` back at 0, and `T` then lands in slot 1. The drain prints `S T`. A trace that places `S` in slot 3 forgot the modulo; one that drains `Q R S T` forgot that `P` and then `Q` and `R` were already served.

```{index} linked queue endpoints and growth choices
```



```{index} Linked Queue Endpoints and Growth Choices
```

## Linked Queue Endpoints and Growth Choices



A singly linked queue uses a head for removal and a tail for insertion. Empty means count zero and both endpoints null. A nonempty chain reaches tail last, and `tail.Next` is null. Removing the only node must clear tail as well as head; otherwise a later enqueue may attach to a detached node.

Unlike removing the last item of a singly linked list, queue dequeue removes the head and needs no predecessor search. Each enqueue/dequeue performs a fixed number of link changes. The program still pays for allocation and memory management.

**Trace:** Draw both endpoints for enqueue A, dequeue A, and enqueue B. Identify the stale-tail bug if dequeue omits the empty case.


In [ ]:
using System;
var tickets = new LinkedQueue<string>();
tickets.Enqueue("A"); tickets.Enqueue("B");
Console.WriteLine(tickets.Dequeue()); Console.WriteLine(tickets.Dequeue());
tickets.Enqueue("C"); Console.WriteLine($"reused={tickets.Dequeue()}, count={tickets.Count}");

public class LinkedQueue<T>
{
    private class Node
    {
        public T Value;
        public Node? Next;
        public Node(T value) => Value = value;
    }
    private Node? head;
    private Node? tail;
    public int Count { get; private set; }
    public void Enqueue(T value)
    {
        var added = new Node(value);
        if (tail is null) head = added;
        else tail.Next = added;
        tail = added;
        Count++;
    }
    public T Peek()
    {
        if (head is null) throw new InvalidOperationException("Queue is empty.");
        return head.Value;
    }
    public T Dequeue()
    {
        T value = Peek();
        head = head!.Next;
        if (head is null) tail = null;
        Count--;
        return value;
    }
}

A
B
reused=C, count=0


```{index} growing a ring and choosing a deque representation
```



```{index} Growing a Ring and Choosing a Deque Representation
```

## Growing a Ring and Choosing a Deque Representation



A growable ring allocates a larger array when full. Copy logical item i from `(front + i) % oldCapacity` into new slot i, then set front to zero. Count stays unchanged; the next insertion slot becomes count. For the full layout `[E, B, C, D]` with front one, growth must produce `[B, C, D, E, unused, ...]`, not `[E, B, C, D, ...]`.

The bounded ring above has O(1) worst-case operation work and no growth. A doubling ring has O(n) worst-case enqueue during resizing and O(1) amortized enqueue. `Queue<T>` provides a growable application collection; bounded buffers need an explicit reject, wait, or overwrite policy.

A deque can use a doubly linked list or a ring with both-end operations. In a ring, adding at the front moves front to `(front - 1 + Capacity) % Capacity`; removing the back uses `(front + Count - 1) % Capacity`. Validate full/empty states first. C# remainder can be negative, which is why the front formula adds capacity.

**Transfer:** Trace add-back A, add-front B, remove-back, remove-front at capacity three. Compare a ring's locality and spare slots with a doubly linked deque's node/link overhead.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Using a `List<T>` as a queue with `Add` and `RemoveAt(0)` gives correct FIFO results. What is wrong with it?

```{dropdown} Answer
Nothing about correctness; everything about cost. `RemoveAt(0)` shifts every remaining element left, so each dequeue is O(n) and a loop that drains n items does about n squared over two moves. A ring or linked queue dequeues in O(1).
```

2. In a ring of capacity 5 with `front` at 3 and `Count` 4, which physical slot holds the third item in service order, and where will the next item be inserted?

```{dropdown} Answer
Logical position i lives at `(front + i) % 5`, so the third item (i = 2) is in slot 0. The next insertion goes to `(3 + 4) % 5`, which is slot 2. Slot 2 is the one free slot.
```

3. A linked queue's `Dequeue` sets `head = head.Next` but never touches `tail`. Describe the sequence of operations that exposes the defect.

```{dropdown} Answer
Enqueue one item, dequeue it, then enqueue another. After the dequeue, `head` is `null` but `tail` still refers to the removed node, so the second enqueue attaches to that detached node and the new item is unreachable from `head`. `Dequeue` must set `tail = null` when the queue becomes empty.
```

4. Why can a bounded ring offer an O(1) worst-case enqueue while a growable one offers only O(1) amortized?

```{dropdown} Answer
The bounded ring never copies; when full it rejects, so every enqueue does constant work. The growable ring must occasionally allocate a larger array and copy every item in logical order, an O(n) step whose cost is spread over the cheap enqueues that follow.
```

```{index} queue; exercise
```



```{index} Exercise
```

## Exercise



Enqueue the ticket IDs `T-101`, `T-102`, and `T-103`, then dequeue and print until the queue is empty. Expected output: the three IDs in the order they were added.


In [ ]:
// Exercise: Serve tickets
// Enqueue three ticket IDs: T-101, T-102, and T-103.
// Then dequeue and print each ticket in service order.

using System;
using System.Collections.Generic;

var tickets = new Queue<string>();

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var tickets = new Queue<string>();
tickets.Enqueue("T-101");
tickets.Enqueue("T-102");
tickets.Enqueue("T-103");

while (tickets.Count > 0)
{
    Console.WriteLine(tickets.Dequeue());
}


T-101
T-102
T-103


```{index} summary
```



```{index} Summary
```

## Summary



Queues model first-come, first-served order. Deques generalize the idea by allowing both ends to participate. Array-backed queues need ring-buffer logic; linked representations make end operations natural when both ends are tracked.


```{rubric} Footnotes
```


[^1502_queue_deque-1]: The .NET base class library includes `Queue<T>` and `LinkedList<T>`, but not a dedicated general-purpose `Deque<T>` type.


[^development-reading]: Further reading: [OpenDSA, Stacks](https://opendsa.org/OpenDSA/Books/Catalog/html/StackArray.html). The worked business scenarios and guided reasoning here are original teaching examples.
